## **1. 라이브러리 Import**

In [ ]:
import pandas as pd
import numpy as np

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 재현성을 위한 seed 설정
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## **2. 데이터 준비**

#### (1) 데이터 로드
#### (2) 컬럼 분류
- key_cols = ['year', 'country', 'admin']
- target_col = 'infection_rate'
- climate_cols = ['max_temperature', 'min_temperature', 'avg_temperature']
- livestock_cols = ['Buffa', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine']
- landuse_cols = ['rate_landuse']
#### (3) X, y 분리
#### (4) Feature scaling
- climate : standard scaling
- livestock, landuse : log transformation + standard scaling

In [18]:
# 데이터 로드
df = pd.read_csv('../data/processed/cleaned_disease_dataset_2015landuse.csv')

print(f"Total samples: {len(df):,}")
df.head()

Total samples: 473


,year,country,admin,infection_rate,max_temperature,min_temperature,avg_temperature,Buffa,Cattl,Chick,Ducks,Goats,Horse,Sheep,Swine,rate_landuse,missing_ratio
0,2010,Burkina Faso,Boucle Du Mouhoun,0.585652,307.42114,297.76172,302.50760,0.0,25.000049,164.065700,0.0,34.153420,0.693863,21.774243,9.682960,0.076206,0.0
1,2010,Burkina Faso,Cascades,0.979432,304.47192,297.88160,300.27893,0.0,38.008688,96.864817,0.0,12.506484,0.008877,12.373391,3.865450,0.074387,0.0
2,2010,Burkina Faso,Centre,1.955671,307.73170,299.50660,302.80447,0.0,17.566445,303.908919,0.0,58.134766,0.019041,40.947445,6.821150,0.407856,0.0
3,2010,Burkina Faso,Centre-est,1.037736,307.41138,298.78003,302.61942,0.0,29.597071,203.094681,0.0,69.010943,0.027864,45.871831,18.292361,0.143950,0.0
4,2010,Burkina Faso,Centre-nord,0.379147,308.31274,297.82104,303.14368,0.0,28.843678,138.295650,0.0,65.134086,0.079677,48.346972,4.834737,0.120649,0.0


In [19]:
# 컬럼 분류
key_cols = ['year', 'country', 'admin']
target_col = 'infection_rate'
climate_cols = ['max_temperature', 'min_temperature', 'avg_temperature']
livestock_cols = ['Buffa', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine']
landuse_cols = ['rate_landuse']
feature_cols = climate_cols + livestock_cols + landuse_cols

print(f"Feature 컬럼 ({len(feature_cols)}개): {feature_cols}")
print(f"Target 컬럼: {target_col}")

Feature 컬럼 (12개): ['max_temperature', 'min_temperature', 'avg_temperature', 'Buffa', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse']
Target 컬럼: infection_rate


In [20]:
# X, y 분리
X = df[feature_cols].copy()
y = df[target_col].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (473, 12)
y shape: (473,)


In [ ]:
# Feature(X) scaling
X_scaled = X.copy()

# Climate : standard normal dist 
# #         => standard scaling
standard_cols = climate_cols
scaler_climate = StandardScaler()
X_scaled[climate_cols] = scaler_climate.fit_transform(X_scaled[climate_cols])

# Livestock, Landuse : right-skewed dist
#                      => log transformation + standard scaling
skewed_cols = livestock_cols + landuse_cols
for col in skewed_cols:
    X_scaled[col] = np.log1p(X_scaled[col])

scaler_skewed = StandardScaler()
X_scaled[skewed_cols] = scaler_skewed.fit_transform(X_scaled[skewed_cols])

In [21]:
print("=== Scaling 전후 비교 ===")
# df에서 mean()을 한 번 취하면 컬럼별 평균이, 두 번 취하면 컬럼 평균의 평균(전체 평균)이 구해짐

print("\n[Scaling 전]")
print(f"평균: {X.mean().mean():.4f}, 표준편차: {X.std().mean():.4f}")

print("\n[Scaling 후]")
print(f"평균: {X_scaled.mean().mean():.4f}, 표준편차: {X_scaled.std().mean():.4f}")

=== Scaling 전후 비교 ===

[Scaling 전]
평균: 97.1898, 표준편차: 76.4891

[Scaling 후]
평균: 0.0000, 표준편차: 0.9176


In [22]:
# train, test dataset 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE
)

print("=== Random Split ===")
print(f"Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTrain - y 평균: {y_train.mean():.4f}, 표준편차: {y_train.std():.4f}")
print(f"Test - y 평균: {y_test.mean():.4f}, 표준편차: {y_test.std():.4f}")

=== Random Split ===
Train set: 378 samples (79.9%)
Test set: 95 samples (20.1%)

Train - y 평균: 5.5219, 표준편차: 6.7059
Test - y 평균: 5.0191, 표준편차: 6.1762
